[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/IyadSultan/CCI/blob/main/session6/student/Lab1_Basic_RAG_LlamaParse_Student.ipynb)


# Lab 1: Basic RAG with LlamaParse
## CCI Session 6

**Duration:** 15 minutes

### Clinical Scenario
> KHCC pediatric oncologists need quick answers from the National Wilms Tumor treatment guidelines (200+ pages with figures, tables, complex layouts). You'll build a RAG system that parses the PDF, embeds it, and answers clinical questions.

### Objective
- Parse a complex clinical PDF with LlamaParse
- Chunk and embed the content
- Store in ChromaDB
- Build a retrieval QA chain
- Test with real clinical questions

---
## Setup — install packages

After installs, **Runtime → Restart session** if Colab still raises import errors.


In [ ]:
!pip install -q llama-parse llama-index langchain langchain-text-splitters langchain-openai langchain-community langchain-chroma chromadb pypdf

## API Keys (Colab `userdata`)

In [ ]:
import os
from google.colab import userdata

os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')
os.environ['LLAMA_CLOUD_API_KEY'] = userdata.get('LLAMA_CLOUD_API_KEY')
print('Keys loaded.')

## Upload `WT.pdf` to Colab
Run the cell, click **Choose Files** and pick the Wilms tumor PDF.

In [ ]:
from google.colab import files
uploaded = files.upload()  # upload WT.pdf
PDF_PATH = '/content/WT.pdf'

---
## Cell 1 — Parse with LlamaParse

Configure **`LlamaParse`** with `result_type='markdown'` so the guideline text keeps structure (tables, headings). Compare visually with PyPDF in the next section.


In [ ]:
from llama_parse import LlamaParse

# TODO: create a LlamaParse parser with result_type='markdown'
# Hint: parser = LlamaParse(api_key=os.environ['LLAMA_CLOUD_API_KEY'], result_type='markdown')
parser = ...

# TODO: parse the PDF
# Hint: documents = parser.load_data(PDF_PATH)
documents = ...

print(f'Number of documents: {len(documents)}')
print('--- Preview ---')
print(documents[0].text[:1500])

In [ ]:
import json
from google.colab import files

# Save parsed markdown to disk so later labs can skip re-parsing
parsed_export = [{'text': d.text, 'metadata': d.metadata} for d in documents]
with open('wt_parsed.json', 'w', encoding='utf-8') as f:
    json.dump(parsed_export, f, ensure_ascii=False)

files.download('wt_parsed.json')
print(f"wt_parsed.json downloaded ({len(parsed_export)} docs) — keep it with WT.pdf and chroma_wt.zip")

## Cell 2 — Compare with naive PyPDF
Notice how tables / figure captions get mangled by the naive loader.

In [ ]:
from langchain_community.document_loaders import PyPDFLoader

# TODO: load with PyPDFLoader
# Hint: naive_docs = PyPDFLoader(PDF_PATH).load()
naive_docs = ...

print(f'Pages (PyPDF): {len(naive_docs)}')
print('--- PyPDF sample (note jumbled tables) ---')
print(naive_docs[5].page_content[:1500])

print('\n--- LlamaParse sample (clean markdown tables) ---')
print(documents[0].text[1500:3000])

In [ ]:
# Cell 2b — Side-by-side parser comparison
# Renders PyPDF (naive) vs LlamaParse (layout-aware) in two HTML columns
# so you can visually judge which one preserves tables, headings, and lists.

from IPython.display import HTML, display
import html

# Pick a region that's likely to contain a table or structured content.
# Adjust these if you want to inspect a different section of the guideline.
PYPDF_PAGE   = 14      # which PyPDF page to show
LLAMA_START  = 0   # char offset into the LlamaParse markdown
LLAMA_LEN    = 4500   # how many chars to show

pypdf_text  = naive_docs[PYPDF_PAGE].page_content[1200:]
llama_text  = documents[14].text[LLAMA_START:LLAMA_START + LLAMA_LEN]

def _panel(title, subtitle, text, bg):
    return f"""
    <div style="flex:1; min-width:0; background:{bg}; border:1px solid #d0d7de;
                border-radius:8px; padding:12px; margin:4px;">
      <div style="font-weight:600; font-size:14px; margin-bottom:2px;">{title}</div>
      <div style="font-size:11px; color:#57606a; margin-bottom:8px;">{subtitle}</div>
      <pre style="white-space:pre-wrap; word-wrap:break-word; font-size:11px;
                  line-height:1.4; max-height:600px; overflow:auto; margin:0;
                  font-family: ui-monospace, SFMono-Regular, Menlo, monospace;">{html.escape(text)}</pre>
    </div>
    """

display(HTML(f"""
<div style="display:flex; flex-wrap:wrap; gap:4px;">
  {_panel("PyPDFLoader (naive)",
          f"page {PYPDF_PAGE} • {len(pypdf_text)} chars • no layout awareness",
          pypdf_text, "#fff5f5")}
  {_panel("LlamaParse (markdown)",
          f"chars {LLAMA_START}–{LLAMA_START+LLAMA_LEN} • layout-aware",
          llama_text, "#f0f7ff")}
</div>
<div style="font-size:11px; color:#57606a; margin-top:8px;">
  Look for: table structure, heading hierarchy, list bullets, column order,
  and whether sentences are broken across the page.
</div>
"""))

---
## Cell 3 — Chunking for RAG

Split the parsed markdown into **chunks** before embedding. Use **`RecursiveCharacterTextSplitter`** so splits prefer paragraph and sentence boundaries.

**Design choices:**
- **`chunk_size`**: target characters per chunk (try 800–1200 for guidelines).
- **`chunk_overlap`**: copied characters between adjacent chunks so facts on boundaries are not lost.
- Alternatives include fixed windows or section-based splitting; this lab uses the recursive splitter.


In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

# Convert LlamaParse docs to LangChain Documents
lc_docs = [Document(page_content=d.text, metadata={'source': 'WT.pdf'}) for d in documents]

# TODO: build a RecursiveCharacterTextSplitter with chunk_size=1000, chunk_overlap=200
splitter = ...

# TODO: split the documents
chunks = ...

print(f'Number of chunks: {len(chunks)}')
print('--- Sample chunk ---')
print(chunks[10].page_content[:600])

---
## Cell 4 — Embedding model

We use **`text-embedding-3-small`** for a good balance of quality and cost. Same model must be used when **building** the index and when **querying** it.

**You could compare:** `text-embedding-3-large` (stronger, pricier) or other providers — always re-embed the whole corpus if you change model.


In [ ]:
from langchain_openai import OpenAIEmbeddings

# TODO: embeddings = OpenAIEmbeddings(model='text-embedding-3-small')
embeddings = ...

---
## Cell 5 — Vector store (Chroma) + export

**Chroma** stores vectors on disk under `persist_directory`. A download cell below will save **`chroma_wt.zip`** — upload it in Labs 2 and 3 instead of re-parsing and re-embedding.

**Other stacks:** FAISS (fast, local), or hosted vector DBs for production multi-user search.

The **retriever** wraps similarity search; `k` is how many chunks the LLM sees as context.

In [ ]:
from langchain_chroma import Chroma

# TODO: create vectorstore from chunks + embeddings, persisted so later labs can reuse it
# Hint: Chroma.from_documents(documents=chunks, embedding=embeddings,
#                              collection_name='wt_good', persist_directory='./chroma_wt')
vectorstore = ...

retriever = vectorstore.as_retriever(search_kwargs={'k': 4})
print(f"Vector store saved → ./chroma_wt  (collection='wt_good')")

In [ ]:
import shutil
from google.colab import files

shutil.make_archive('chroma_wt', 'zip', '.', 'chroma_wt')
files.download('chroma_wt.zip')
print("chroma_wt.zip downloaded — keep it alongside WT.pdf to reuse in Labs 2 and 3")

In [ ]:
# ── Save to Google Drive (persistent — no re-upload needed in future sessions) ───────────────────
from google.colab import drive
import os, shutil

drive.mount('/content/drive')
DRIVE_DIR = '/content/drive/MyDrive/CCI_Session6'
os.makedirs(DRIVE_DIR, exist_ok=True)

# Combined markdown — human-readable, searchable directly in Drive
combined_md = '\n\n---\n\n'.join(d.text for d in documents)
with open(f'{DRIVE_DIR}/WT_parsed.md', 'w', encoding='utf-8') as f:
    f.write(combined_md)

# Copy JSON + Chroma zip (already written to /content above) → Drive
shutil.copy('wt_parsed.json', f'{DRIVE_DIR}/wt_parsed.json')
shutil.copy('chroma_wt.zip',  f'{DRIVE_DIR}/chroma_wt.zip')

print(f"Saved to Google Drive → {DRIVE_DIR}")
print(f"  WT_parsed.md   ({len(combined_md):,} chars) — parsed markdown")
print(f"  wt_parsed.json — LangChain Document list (Labs 2–3)")
print(f"  chroma_wt.zip  — vector store (Labs 2–3)")
# ─────────────────────────────────────────────────────────────────────────────────────────────────

## Cell 6 — Build RAG Chain

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.runnables import RunnableLambda
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

llm = ChatOpenAI(model='gpt-4o-mini', temperature=0)

prompt = ChatPromptTemplate.from_messages([
    ('system', 'You are a clinical assistant. Answer the question using ONLY the provided context. If unknown, say so.\n\nContext:\n{context}'),
    ('human', '{input}')
])

def format_docs(docs):
    return "\n\n".join(d.page_content for d in docs)

# TODO: build a RAG chain using LCEL
# It must accept {'input': question} and return {'answer': ..., 'context': [docs]}
# Hint:
#   def _rag(d):
#       docs = retriever.invoke(d['input'])
#       answer = (prompt | llm | StrOutputParser()).invoke({'context': format_docs(docs), 'input': d['input']})
#       return {'answer': answer, 'context': docs}
#   rag_chain = RunnableLambda(_rag)
rag_chain = ...

q1 = 'What is the standard treatment for Stage III favorable histology Wilms tumor?'
q2 = 'What are the most common side effects of vincristine?'

for q in [q1, q2]:
    print(f'\nQ: {q}')
    res = rag_chain.invoke({'input': q})
    print('A:', res['answer'])

## Cell 7 — Inspect Retrieved Chunks (Citations)

In [ ]:
res = rag_chain.invoke({'input': q1})
for i, doc in enumerate(res['context']):
    print(f'--- Chunk {i+1} ---')
    print(doc.page_content[:300])
    print()

## Stretch Challenge
Try `chunk_size` of 500 and 2000. Re-run question 1. Which produces better citations?

## KHCC Connection
This pattern is the foundation for any internal KHCC document QA system: clinical pathways, drug formulary, IRB SOPs. LlamaParse keeps tables and figures intact — critical for guideline lookup.